## Repository path configuration
All repository files are accessed through relative paths. Run the notebook from the repository root or from the `notebooks/` directory.


In [ ]:
from pathlib import Path
import os

# Resolve the repository root whether Jupyter starts in the repository root
# or directly inside the notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

DATA_DIR = ROOT / "data"
EXTERNAL_DATA_DIR = DATA_DIR / "external"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = ROOT / "outputs"
RESULTS_DIR = OUTPUTS_DIR / "results"
FIGURES_DIR = OUTPUTS_DIR / "figures"
EXTERNAL_MATERIALS_DIR = ROOT / "external_materials"
MODEL_WEIGHTS_DIR = EXTERNAL_MATERIALS_DIR / "model_weights"

for folder in [
    EXTERNAL_DATA_DIR,
    PROCESSED_DATA_DIR,
    PROCESSED_DATA_DIR / "logs",
    RESULTS_DIR,
    FIGURES_DIR,
    MODEL_WEIGHTS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)


In [ ]:
# Google Drive mounting is intentionally not used in this repository.

In [ ]:
import os
os.environ["WANDB_API_KEY"] = os.getenv("WANDB_API_KEY", "")

In [ ]:
!pip install wandb -qU  # Εγκατάσταση της πιο πρόσφατης έκδοσης της βιβλιοθήκης WandB.
import wandb            # Εισαγωγή της WandB.

# Αυτή η εντολή ενεργοποιεί τον χρήστη να συνδεθεί στο λογαριασμό του WandB.
# Θα χρειαστεί ένα API Key.
wandb.login(key=os.environ["WANDB_API_KEY"]) if os.getenv("WANDB_API_KEY") else None

In [ ]:
"""
Εγκατάσταση απαραίτων στοιχείων.
"""
!pip install accelerate -U
!pip install transformers[torch]
!pip install bitsandbytes
!pip install -qU peft  # For LoRA integration
!pip install -qU datasets

In [ ]:
import torch
import nltk
import pandas as pd
import numpy as np
import string
import random
import seaborn as sns
import matplotlib.pyplot as plt
import torch.nn as nn
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, BitsAndBytesConfig
from transformers import AutoModelForCausalLM
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight   # <-- ΠΡΟΣΘΗΚΗ
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.stem import WordNetLemmatizer
#from huggingface_hub import login
from peft import get_peft_model, LoraConfig, TaskType
from transformers import TrainerCallback
import bitsandbytes as bnb
import time
from torch.utils.data import DataLoader, WeightedRandomSampler  # <-- ΝΕΟ

In [ ]:
# Κατέβασμα των απαραίτητων δεδομένων από το nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')  # Για σωστό lemmatization
nltk.download('averaged_perceptron_tagger')  # POS tagging
nltk.download('punkt')  # Tokenizer
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
# Εισαγωγή του Access Token απο την Hugging face.
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

In [ ]:
# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Function to clean text
def get_wordnet_pos(word):
    tag = pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

def clean_text(text):
    stop_words = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    words = word_tokenize(text)
    filtered_words = [word for word in words if word not in stop_words]
    lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in filtered_words]
    return " ".join(lemmatized_words)

In [ ]:
# Load dataset
df = pd.read_csv("data/external/sentences_dataset_45269.csv")

In [ ]:
"""print("\nΠροτάσεις πριν τον καθαρισμό!\n")
print(df.head(30))

df["sentence"] = df["sentence"].apply(clean_text)

print("\nΠροτάσεις μετά τον καθαρισμό!\n")
print(df.head(30))"""

In [ ]:
# Αφαίρεση διπλότυπων εγγραφών. Διατηρείται μόνο η πρώτη εμφάνιση της εν λόγω γραμμής.
# Υπολογισμός αριθμού διπλών εγγραφών.
num_duplicates = df['sentence'].duplicated().sum()
print("Πλήθος διπλών εγγραφών στη στήλη sentence:", num_duplicates)

# Αποθήκευση του αρχικού πλήθους γραμμών.
before = len(df)

# Αφαίρεση διπλότυπων (κρατάμε την πρώτη εμφάνιση).
df = df.drop_duplicates(subset='sentence', keep='first')

# Υπολογισμός του πόσες γραμμές διαγράφηκαν.
after = len(df)
deleted = before - after
print("Πλήθος εγγραφών που διαγράφηκαν:", deleted)


# Υπολογισμός στατιστικών για τη στήλη polarity.
polarity_stats = df['polarity'].value_counts()

# Εμφάνιση των αποτελεσμάτων.
print("Polarity Statistics:\n", polarity_stats)

In [ ]:
# Split dataset
train_data, temp_data = train_test_split(df, test_size=0.3, stratify=df['polarity'], random_state=SEED)
val_data, test_data = train_test_split(temp_data, test_size=0.5, stratify=temp_data['polarity'], random_state=SEED)

# Εμφάνιση των διαστάσεων των σετ δεδομένων.
print("Διαστάσεις Σετ Εκπαίδευσης:", train_data.shape)
print("Διαστάσεις Σετ Ελέγχου:", test_data.shape)

In [ ]:
# Model name
MODEL_NAME = "deepseek-ai/deepseek-llm-7b-base"
#MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
#MODEL_NAME = "real-jiakai/DeepSeek-R1-Distill-Qwen-7B-News-Classifier"

EPOCHS = int(input("Δώσε αριθμό εποχών εκπαίδευσης "))

# Δεκτές εποχές απο 3 έως 20.
while not EPOCHS >= 3 or not EPOCHS <=20:
    EPOCHS = int(input("Δώσε αριθμό εποχών εκπαίδευσης "))

In [ ]:
# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [ ]:
# Load tokenizer and model.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=True, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # Ορισμός padding token.

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=True,
    quantization_config=bnb_config,
    num_labels=3,
    device_map="auto",
    trust_remote_code=True,
    output_hidden_states=True
)

# Καθορισμός pad_token_id για να αναγνωρίζεται από το μοντέλο.
base_model.config.pad_token_id = tokenizer.pad_token_id

# LoRA fine-tuning για sequence classification, όχι CAUSAL_LM. Οπότε έβαλα task_type=TaskType.SEQ_CLS.
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "up_proj", "down_proj", "gate_proj"
    ]  # Προσαρμογή μόνο των κατάλληλων layers
)

base_model = get_peft_model(base_model, lora_config)

# Print trainable parameters.
base_model.print_trainable_parameters()

In [ ]:
# Tokenization
def tokenize_function(examples):
    return tokenizer(examples, padding="max_length", truncation=True, max_length=256)

train_encodings = tokenize_function(train_data['sentence'].tolist())
test_encodings = tokenize_function(test_data['sentence'].tolist())
val_encodings = tokenize_function(val_data['sentence'].tolist())

In [ ]:
# Dataset class
class PolarityDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = PolarityDataset(train_encodings, torch.tensor(train_data['polarity'].tolist()))
test_dataset = PolarityDataset(test_encodings, torch.tensor(test_data['polarity'].tolist()))
val_dataset = PolarityDataset(val_encodings, torch.tensor(val_data['polarity'].tolist()))

In [ ]:
# =========================
# WeightedRandomSampler για class imbalance  (ΝΕΟ)
# =========================
train_labels_np = train_dataset.labels.numpy()
class_sample_counts = np.bincount(train_labels_np)
print("Samples per class (train):", class_sample_counts)

weights_per_class = 1.0 / class_sample_counts
print("Weights per class:", weights_per_class)

sample_weights = weights_per_class[train_labels_np]
sample_weights = torch.DoubleTensor(sample_weights)

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

In [ ]:
# -------------------------
# Class weights για το loss (CrossEntropy)
# -------------------------
class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_labels_np
)
print("Class weights (for loss):", class_weights_np)

class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float32)


In [ ]:
class DeepSeekForClassification(nn.Module):
    def __init__(self, base_model, num_labels, class_weights=None):
        super().__init__()
        self.base_model = base_model
        hidden_size = base_model.config.hidden_size

        self.classifier = nn.Linear(hidden_size, num_labels)

        if class_weights is not None:
            self.register_buffer("class_weights", class_weights)
        else:
            self.class_weights = None

        self.loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights
        ) if class_weights is not None else nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        hidden_states = outputs.hidden_states[-1]   # [B, T, H]

        # --- MEAN POOLING ---
        if attention_mask is None:
            pooled = hidden_states.mean(dim=1)
        else:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)

        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            labels = labels.to(torch.long)
            cw = None
            if self.class_weights is not None:
                cw = self.class_weights.to(logits.device)
                loss_fct = nn.CrossEntropyLoss(weight=cw)
            else:
                loss_fct = nn.CrossEntropyLoss()

            loss = loss_fct(logits, labels)

        return {"loss": loss, "logits": logits}


# DeepSeek με κεφαλή ταξινόμησης + class weights για το loss
model = DeepSeekForClassification(
    base_model,
    num_labels=3,
    class_weights=class_weights_tensor
)

In [ ]:
from transformers import EarlyStoppingCallback

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    run_name="Sentiment_Analysis_DeepSeek",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    #warmup_steps=100,
    warmup_ratio = 0.05,
    weight_decay=0.02,
    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    save_safetensors=False,
    load_best_model_at_end=True,
    gradient_accumulation_steps=8,
    save_total_limit=2,
    learning_rate=2e-5,
    fp16=True,
    report_to="none", # να μην γράφει σε wandb
    greater_is_better=False,
    optim="adamw_torch_fused", # γρήγορος και σταθερός optimizer
    metric_for_best_model="eval_loss",
    max_grad_norm=1.0  # Προσθήκη gradient clipping.
)

#my_optimizer = bnb.optim.Adam32bit(model.parameters(), lr=5e-6) #2e-5

class LossLoggerCallback(TrainerCallback):
    def __init__(self):
        # θα κρατήσουμε όλα τα logs με epoch, loss, eval_loss
        self.records = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        epoch = logs.get("epoch", None)
        loss = logs.get("loss", None)
        eval_loss = logs.get("eval_loss", None)

        # κρατάμε μόνο logs που έχουν epoch ΚΑΙ τουλάχιστον ένα από loss / eval_loss
        if epoch is not None and (loss is not None or eval_loss is not None):
            self.records.append({
                "epoch": epoch,
                "loss": loss,
                "eval_loss": eval_loss
            })

# Δημιουργία callback
loss_logger = LossLoggerCallback()

In [ ]:
# =========================
# Custom Trainer με WeightedRandomSampler  (ΝΕΟ)
# =========================
class WeightedTrainer(Trainer):
    def get_train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.per_device_train_batch_size,
            sampler=train_sampler,
            collate_fn=self.data_collator,
        )

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    #optimizers=(my_optimizer, None),
    callbacks=[loss_logger, EarlyStoppingCallback(early_stopping_patience=4)]
)

# Ορισμός του χρόνου έναρξης.
start_time = time.time()

# Εκπαίδευση του μοντέλου.
print("Ξεκινά η εκπαίδευση του μοντέλου...\n")
#print("Το deepseek δεν κάνει σωστή καταγραφή των εποχών. Μόλις ολοκληρωθεί η 2η εποχή πάει και την καταγράφει\n")
#print("στην πρώτη γραμμή με αποτέλεσμα να αντικαθιστά την 1η εποχή. Οπότε λαμβάνουμε υπόψιν μόνο την γραφική παράσταση κάτω.")


if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()


# Train model
trainer.train()

# Ορισμός του χρόνου λήξης
end_time = time.time()

# Υπολογισμός του συνολικού χρόνου εκπαίδευσης
total_time = end_time - start_time

# Μετατροπή σε λεπτά & δευτερόλεπτα.
minutes = int(total_time // 60)
seconds = int(total_time % 60)
print(f"\nΣυνολικός Χρόνος Εκπαίδευσης: {minutes} λεπτά και {seconds} δευτερόλεπτα ({training_args.num_train_epochs} εποχές)")

# Τα raw logs από το callback
records = loss_logger.records

df_logs = pd.DataFrame(records)

# Διαχωρίζουμε train και val
train_df = df_logs[df_logs["loss"].notna()][["epoch", "loss"]]
train_df = train_df.rename(columns={"loss": "train_loss"})

eval_df = df_logs[df_logs["eval_loss"].notna()][["epoch", "eval_loss"]]
eval_df = eval_df.rename(columns={"eval_loss": "val_loss"})

# Κάνουμε merge ανά epoch
df_loss = pd.merge(train_df, eval_df, on="epoch", how="left")

# Τακτοποιούμε / στρογγυλοποιούμε
df_loss = df_loss.sort_values("epoch").reset_index(drop=True)
df_loss["train_loss"] = df_loss["train_loss"].astype(float)
df_loss["val_loss"] = df_loss["val_loss"].astype(float)

print(df_loss)

# Αποθήκευση σε CSV για αυτό το μοντέλο
NAME = "DeepSeek"  # ή "ELECTRA", "ALBERT", κτλ
csv_path = f"data/processed/logs/loss_{NAME}.csv"
df_loss.to_csv(csv_path, index=False)
print("Saved loss CSV to:", csv_path)

In [ ]:
# Save model and tokenizer
torch.save(model.state_dict(), 'external_materials/model_weights/DeepSeek/saved_model/saved_deepSeek_state_dict.pth')
tokenizer.save_pretrained('external_materials/model_weights/DeepSeek/saved_tokenizer')

# Predictions
predictions = trainer.predict(test_dataset)
predicted_classes = np.argmax(predictions.predictions, axis=-1)

# Classification report
report = classification_report(test_data['polarity'], predicted_classes, target_names=['Negative', 'Neutral', 'Positive'])
print(report)

with open(f"outputs/results/classification_reports/{NAME}_classification_report.txt", "w", encoding="utf-8") as f:
    f.write(report)

# Ενοποιημένο μέγεθος γραμματοσειράς
plt.rcParams.update({'font.size': 11})
plt.figure(figsize=(6, 4))

# Confusion Matrix
cm = confusion_matrix(test_data['polarity'], predicted_classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Neutral', 'Positive'], yticklabels=['Negative', 'Neutral', 'Positive'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()

plt.savefig(
    f'outputs/figures/fig/{NAME}_heatmap_SA_citation.pdf',
    format='pdf',
    bbox_inches='tight',
    dpi=300
)

plt.show()

In [ ]:
################################